In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
#

In [ ]:
data_dir = 'C:\\Users\\User\\Projects\\ML-FINAL-PROJECT\\data\\'
adult = pd.read_csv(data_dir + 'adult.data', header=None)
adult.head(5)

In [ ]:
adult.shape

In [ ]:
adult.columns

In [ ]:
adult = adult.rename(columns={0:'Age', 1:'Workclass', 2:'Fnlwgt', 3:'Education', 4:'Education_num',
                              5:'marital_status', 6:'Occupation', 7:'Relationship',
                              8:'Race', 9:'Sex', 10:'capital_gain', 11:'capital_loss', 12:'hours_per_week',
                              13:'Country', 14:'Income'})
adult.head(5)

In [ ]:
adult['Education_num'].value_counts()

In [ ]:
adult['Education'].value_counts()

In [ ]:
adult = adult.drop(columns=['Education'])

In [ ]:
adult['Income'].value_counts()

In [ ]:
adult['Income'] = adult['Income'].str.strip()
adult['Income'] = adult['Income'].replace(['<=50K', '>50K'], [0, 1]).astype(int)
adult['Income'].value_counts()

In [ ]:
adult['Sex'].value_counts()

In [ ]:
adult['Sex'] = adult['Sex'].str.strip().map({'Male': 1, 'Female': 0})
print("Value counts for encoded 'Sex' column:")
display(adult['Sex'].value_counts())

In [ ]:
print("Number of duplicates: ", adult.duplicated().sum())
adult.drop_duplicates(inplace=True)

In [ ]:
categorical_cols = adult.select_dtypes(include='object').columns

for col in categorical_cols:
    if ' ?' in adult[col].unique():
        print(f"  -> Found '?' in column '{col}'. This might indicate missing values.")

In [ ]:
import numpy as np

for col in ['Workclass', 'Occupation', 'Country']:
    adult[col] = adult[col].replace(' ?', np.nan)
    # Impute missing values with the mode
    mode_value = adult[col].mode()[0]
    adult[col] = adult[col].fillna(mode_value) # Fixed: Removed inplace=True and assigned back
    print(f"Missing values in '{col}' after imputation: {adult[col].isnull().sum()}")

In [ ]:
numerical_cols = adult.select_dtypes(include=np.number).columns

plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1) # Adjust subplot grid as needed
    adult[col].hist(bins=20)
    plt.title(col)
    plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
adult.describe()

In [ ]:
print("Unique values and their counts for categorical columns:")
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(adult[col].value_counts())

In [ ]:
def group_marital(status):
    status = str(status).strip()
    if 'Married' in status:
        return 2  # married
    elif 'Never-married' in status:
        return 0  # Never-married
    else:
        return 1  # other

adult['marital_status'] = adult['marital_status'].apply(group_marital)

In [ ]:
occ_counts = adult['Occupation'].value_counts()
adult['Occupation'] = adult['Occupation'].map(occ_counts)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Apply label encoding to 'Relationship' column
adult['Relationship'] = LabelEncoder().fit_transform(adult['Relationship'].astype(str))

In [ ]:
# if country is us 1, otherwise 0
adult['is_US'] = adult['Country'].apply(lambda x: 1 if str(x).strip() == 'United-States' else 0)

# Drop it
adult.drop('Country', axis=1, inplace=True)

In [ ]:
print(f"Shape before re-encoding: {adult.shape}")

categorical_cols = adult.select_dtypes(include='object').columns

# Apply one-hot encoding to remaining categorical features
adult_encoded = pd.get_dummies(adult, columns=categorical_cols, drop_first=True)

print(f"Shape after one-hot encoding: {adult_encoded.shape}")
display(adult_encoded.head())

In [ ]:
categorical_cols = adult.select_dtypes(include='object').columns

plt.figure(figsize=(20, 25))
for i, col in enumerate(categorical_cols):
    plt.subplot(6, 3, i + 1) # Adjust subplot grid as needed
    sns.countplot(data=adult, x=col, palette='viridis')
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
plt.show()

### Feature Selection and Correlation Analysis

Now, we'll examine the correlation between the numerical features and the target variable (`Income`) to identify the most predictive features and check for multicollinearity.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate correlation matrix for numerical features and Income
corr_matrix = adult.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

### Preparing X and y for Modeling

We will separate the features ($X$) and the target ($y$). We'll also drop `Fnlwgt` as it is an administrative weight and typically doesn't hold predictive value for individual income levels in this context.

In [ ]:
# Define target and features
y_train = adult_encoded['Income']

# Dropping Fnlwgt as it is usually not a useful predictor for this dataset
X_train = adult_encoded.drop(columns=['Income', 'Fnlwgt'])

### Feature Scaling

To ensure our model treats all features fairly regardless of their original magnitude, we will scale the numerical columns using `StandardScaler`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns in the feature set (excluding the one-hot encoded ones)
num_cols = ['Age', 'Education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

scaler = StandardScaler()

# Fit on training data and transform both training and testing sets
X_train_scaled = X_train.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])

X_train_scaled.head(5)

### Exporting Processed Data

Finally, we save the prepared training and testing sets to CSV files for use in the modeling phase.

In [ ]:
# Saving processed datasets
X_train_scaled.to_csv('adult_X_train_processed.csv', index=False)
y_train.to_csv('adult_y_train_processed.csv', index=False)

print("Datasets saved successfully: X_train_processed.csv, y_train.csv")

## Preprocessing adult.test Dataset

### Step 1: Load adult.test

In [ ]:
adult_test = pd.read_csv('../data-2/adult.test', header=None, skiprows=1)
print(f"Original shape: {adult_test.shape}")
adult_test.head()

### Step 2: Rename columns to match training data

In [ ]:
adult_test = adult_test.rename(columns={0:'Age', 1:'Workclass', 2:'Fnlwgt', 3:'Education', 4:'Education_num',
                              5:'marital_status', 6:'Occupation', 7:'Relationship',
                              8:'Race', 9:'Sex', 10:'capital_gain', 11:'capital_loss', 12:'hours_per_week',
                              13:'Country', 14:'Income'})
adult_test.head(5)

### Step 3: Drop Education column

In [ ]:
adult_test = adult_test.drop(columns=['Education'])
print(f"Shape after dropping Education: {adult_test.shape}")

### Step 4: Encode Income (remove trailing period and convert to binary)

In [ ]:
adult_test['Income'] = adult_test['Income'].str.strip().str.rstrip('.')
adult_test['Income'] = adult_test['Income'].replace(['<=50K', '>50K'], [0, 1]).astype(int)
print("Income value counts:")
print(adult_test['Income'].value_counts())

### Step 5: Encode Sex (Male=1, Female=0)

In [ ]:
adult_test['Sex'] = adult_test['Sex'].str.strip().map({'Male': 1, 'Female': 0})
print("Sex value counts:")
print(adult_test['Sex'].value_counts())

### Step 6: Remove duplicates

In [ ]:
print(f"Number of duplicates: {adult_test.duplicated().sum()}")
adult_test.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {adult_test.shape}")

### Step 7: Handle missing values (replace '?' with mode)

In [ ]:
for col in ['Workclass', 'Occupation', 'Country']:
    adult_test[col] = adult_test[col].replace(' ?', np.nan)
    mode_value = adult_test[col].mode()[0]
    adult_test[col] = adult_test[col].fillna(mode_value)
    print(f"Missing values in '{col}' after imputation: {adult_test[col].isnull().sum()}")

### Step 8: Group marital status

In [ ]:
adult_test['marital_status'] = adult_test['marital_status'].apply(group_marital)
print("Marital status value counts:")
print(adult_test['marital_status'].value_counts())

### Step 9: Encode Occupation (map to frequency counts)

In [ ]:
occ_counts = adult_test['Occupation'].value_counts()
adult_test['Occupation'] = adult_test['Occupation'].map(occ_counts)

### Step 10: Label encode Relationship

In [ ]:
adult_test['Relationship'] = LabelEncoder().fit_transform(adult_test['Relationship'].astype(str))

### Step 11: Create is_US feature and drop Country

In [ ]:
adult_test['is_US'] = adult_test['Country'].apply(lambda x: 1 if str(x).strip() == 'United-States' else 0)
adult_test.drop('Country', axis=1, inplace=True)

# One-hot encode remaining categorical features
categorical_cols = adult_test.select_dtypes(include='object').columns
adult_test_encoded = pd.get_dummies(adult_test, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {adult_test_encoded.shape}")
display(adult_test_encoded.head())

### Step 12: Scale numerical features and prepare for evaluation

In [ ]:
# Separate target and features
y_test = adult_test_encoded['Income']
X_test = adult_test_encoded.drop(columns=['Income', 'Fnlwgt'])

# Scale numerical features
scaler_test = StandardScaler()
X_test_scaled = X_test.copy()
X_test_scaled[num_cols] = scaler_test.fit_transform(X_test[num_cols])

# Save processed test datasets
X_test_scaled.to_csv('adult_X_test_processed.csv', index=False)
y_test.to_csv('adult_y_test_processed.csv', index=False)

print(f"Test set shape: X_test={X_test_scaled.shape}, y_test={y_test.shape}")
print("\nTest datasets saved successfully:")
print("  - adult_X_test_processed.csv")
print("  - adult_y_test_processed.csv")
X_test_scaled.head()

In [ ]:
print("Shape: ", X_train.shape)
print("Shape test: ", X_test.shape)